# Notebook funcional - Ingesta

Notebook simple para cargar fuentes a Qdrant sin pasar por consola.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

src_path = Path.cwd().resolve().parent  
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

load_dotenv(src_path / ".env")
os.chdir(src_path)

In [2]:
import logging

from ingestion.embedder import Embedder
from ingestion.run_ingestion import ingest_breb, ingest_products, ingest_reviews
from storage.qdrant_store import QdrantStore

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)

qdrant_host = os.getenv('QDRANT_HOST', 'localhost').strip() or 'localhost'
qdrant_port = int(os.getenv('QDRANT_PORT', '6333'))

store = QdrantStore(host=qdrant_host, port=qdrant_port)
embedder = Embedder()

print(f'Qdrant: {qdrant_host}:{qdrant_port}')
print(f'Embedding model: {embedder.model}')

Qdrant: localhost:6333
Embedding model: text-embedding-3-large


2026-03-30 09:05:56 | INFO | httpx | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


In [ ]:
# Cambiar el valor de la fuente dependiendo la colleccion que se desee cargar: reviews, products, breb o all
source = 'all'

if source == 'reviews':
    total = ingest_reviews(store, embedder)
    print(f'Ingesta lista para reviews. Total indexado: {total}')
elif source == 'products':
    total = ingest_products(store, embedder)
    print(f'Ingesta lista para products. Total indexado: {total}')
elif source == 'breb':
    total = ingest_breb(store, embedder)
    print(f'Ingesta lista para breb. Total indexado: {total}')
elif source == 'all':
    totals = {
        'reviews': ingest_reviews(store, embedder),
        'products': ingest_products(store, embedder),
        'breb': ingest_breb(store, embedder),
    }
    print('Ingesta completa:', totals)
else:
    raise ValueError("source debe ser reviews, products, breb o all")